In [1]:
import pandas as pd
import numpy as np

# Methodology

Sentiment labels are assigned to each news article based on the sign of this aggregated `three-day excess return`. This excess return is calculated from the day a news article is first published and extends over the two subsequent days. To elaborate, excess return is defined as the difference between the return of a particular stock and the overall market return on the same day. This calculation is not limited to the day the news is published; instead, it aggregates the returns for the following two days as well, providing a
comprehensive three-day outlook.

A positive aggregated excess return leads to a sentiment label of `1`, indicating a positive sentiment. Conversely, a non-positive aggregated excess return results in a sentiment label of `0`, suggesting a negative sentiment.

In [61]:
stocks = pd.ExcelFile("./Data/stocks_data.xlsx")

stocks_dict = {
    sheet_name: stocks.parse(sheet_name) for sheet_name in stocks.sheet_names
}

In [76]:
def excess_return_sentiment_label(stock_data, market_data, num_days=3, price_used="Adj Close"):
    # Make a copy so as not to modify original
    stock_df = stock_data.copy()
    market_df = market_data.copy()

    # Set date as index
    stock_df = stock_df.set_index("date")
    market_df = market_df.set_index("date")

    # Filter for necessary price columns
    stock_df = stock_df[[price_used]].copy()
    market_df = market_df[[price_used]].copy()
    market_df = market_df.add_prefix("SPY_")

    # Calculate daily returns
    merged_df = stock_df.join(market_df)
    merged_df = merged_df.pct_change()

    # Calculate specified X-days aggregated returns
    merged_df["excess_returns"] = merged_df["Adj Close"] - merged_df["SPY_Adj Close"]
    # merged_df["X_days_excess_returns"] =\
    #     (
    #         merged_df["excess_returns"]
    #         .rolling(num_days)
    #         .sum()
    #         .shift(-(num_days - 1))
    #     )
    
    merged_df["X_days_excess_returns"] =\
        (
            (1 + merged_df["excess_returns"])
            .rolling(num_days)
            .apply(lambda x: x.cumprod()[-1], raw=True)
            .shift(-(num_days - 1)) 
            - 1
        )
    # Create sentiment label
    merged_df["sentiment_label"] = np.sign(merged_df["X_days_excess_returns"])

    return merged_df

In [77]:
excess_return_sentiment_label(stocks_dict['AAPL'], stocks_dict['SPY']).head(10)

,Adj Close,SPY_Adj Close,excess_returns,X_days_excess_returns,sentiment_label
date,,,,,
2023-01-03,NaN,NaN,NaN,NaN,NaN
2023-01-04,0.010314,0.007720,0.002594,0.017314,1.0
2023-01-05,-0.010605,-0.011413,0.000809,0.019406,1.0
2023-01-06,0.036794,0.022932,0.013862,0.015979,1.0
2023-01-09,0.004089,-0.000567,0.004656,0.010570,1.0
2023-01-10,0.004456,0.007013,-0.002556,0.001621,1.0
2023-01-11,0.021112,0.012648,0.008464,0.010454,1.0
2023-01-12,-0.000599,0.003641,-0.004240,0.012582,1.0
2023-01-13,0.010119,0.003879,0.006240,0.027488,1.0


In [78]:
news = pd.ExcelFile('./Data/news_data.xlsx')
news_dict = {
    sheet_name: news.parse(sheet_name) for sheet_name in news.sheet_names
}

In [79]:
labelled_news_dict = {}

for k, v in news_dict.items():

    try:
        # Prepare the excess_returns dataframe
        excess_returns_df = excess_return_sentiment_label(
            stocks_dict[k], stocks_dict["SPY"]
        )

        # Set date column of news dataframe to datetime
        temp_df = v.copy()
        temp_df["date"] = pd.to_datetime(temp_df["date"]).dt.normalize()
        temp_df["sentiment_label"] = temp_df["date"].apply(
            lambda x: (
                excess_returns_df.loc[x]["sentiment_label"]
                if x in excess_returns_df.index
                else np.nan
            )
        )

        labelled_news_dict[k] = temp_df
        
    except:
        pass